# Alpha Search — AI Infrastructure & Semiconductor Universe
## Risk-Adjusted Alpha Study

**Research question:** Does a liquid AI-infrastructure + semiconductor equity universe produce
risk-adjusted alpha *after transaction costs*, using pre-registered cross-sectional momentum,
trend-following, mean-reversion, and Donchian-breakout signals?

**Maps to repo:** `notebooks/02_ai_infrastructure_alpha_research_colab.ipynb`

---

### Integrity rules baked into this notebook (do not relax them):
1. **No synthetic data.** All prices come from Yahoo Finance (real OHLCV).
2. **No Sharpe-fishing.** Parameters are declared before any results are observed.
3. **Costs are never hidden.** Net (after-cost) metrics are the headline.
4. **No FRED / no macro.** rf = 0.0 — flagged where it matters.
5. **No look-ahead.** `shift(1)` before `rolling()` throughout.

## 1. Research Disclaimer

> **⚠ IMPORTANT — READ BEFORE PROCEEDING**
>
> This notebook is for **research and educational purposes only.**
> It does **not** constitute investment advice.
> **Past performance does not guarantee future results.**
>
> Backtests are subject to survivorship bias, look-ahead risk, and model risk.
> rf = 0.0 (no FRED) slightly **overstates** long-only Sharpe in high-rate regimes.
> Negative results are reported as-is — no parameters are tuned to improve them.
>
> Always consult a licensed financial advisor before making investment decisions.

## 2. Install Alpha Search

In [ ]:
import sys
try:
    import alpha_search
    print(f"alpha_search already installed")
except ImportError:
    import os
    if not os.path.exists('alpha-search'):
        !git clone https://github.com/alpha-search/alpha-search.git
    %cd alpha-search
    !pip install -q -e .
    !pip install -q yfinance>=0.2.48 matplotlib>=3.8
    print("Installed.")

## 3. Imports

In [ ]:
from __future__ import annotations
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

from alpha_search.research.ai_infra_strategy_pipeline import (
    get_ai_infra_universe,
    download_ai_infra_data,
    validate_ai_infra_data,
    build_cross_sectional_momentum_signal,
    build_trend_following_signal,
    build_mean_reversion_signal,
    build_breakout_signal,
    build_monthly_rebalance_weights,
    run_strategy_backtest,
    calculate_strategy_metrics,
    calculate_alpha_beta_vs_benchmark,
    export_ai_infra_outputs,
    generate_ai_infra_report,
    run_ai_infra_research,
)

warnings.filterwarnings('ignore', category=FutureWarning)
matplotlib.rcParams['figure.dpi'] = 110
pd.set_option('display.float_format', lambda v: f'{v:.4f}')
print('Imports complete.')

## 4. Define AI Infrastructure Universe

Pre-defined universe: US-listed AI-infra + semiconductor equities with sufficient liquidity.
Screened per-rebalance on trailing median daily dollar volume (≥ $25M).
No survivorship-bias hand-picking — tickers with short history are simply not selected
until they qualify.

In [ ]:
universe = get_ai_infra_universe()
all_symbols = [t for tickers in universe.values() for t in tickers]

print(f'Total universe: {len(all_symbols)} symbols')
for segment, tickers in universe.items():
    print(f'  {segment:20s}: {tickers}')

# === CONFIGURATION (pre-registered — do not tune after seeing results) ===
PERIOD           = '5y'
INTERVAL         = '1d'
COST_BPS         = 10.0    # one-way commission
SLIPPAGE_BPS     = 10.0    # one-way slippage
TOTAL_COST_BPS   = COST_BPS + SLIPPAGE_BPS
LONG_ONLY        = False   # True = suppress short leg
PRIMARY_BENCH    = 'SOXX'
OUTPUT_DIR       = 'outputs/research_runs/ai_infrastructure'

print(f'\nPeriod: {PERIOD} | Interval: {INTERVAL}')
print(f'Costs: {COST_BPS}+{SLIPPAGE_BPS} = {TOTAL_COST_BPS} bps round-trip')
print(f'rf = 0.0 (no FRED — overstates long-only Sharpe in high-rate regimes)')

## 5. Fetch Real yfinance Data

In [ ]:
print(f'Downloading {len(all_symbols)} universe symbols + 4 benchmarks...')
close, volume, bench_close = download_ai_infra_data(
    symbols=all_symbols,
    period=PERIOD,
    interval=INTERVAL,
)

print(f'\nClose shape  : {close.shape}')
print(f'Date range   : {close.index.min().date()} → {close.index.max().date()}')
print(f'Benchmarks   : {list(bench_close.columns)}')
print(f'\nSample (last 3 rows):')
close.tail(3)

## 6. Validate OHLCV

In [ ]:
all_valid, val_report, skipped, valid_close, valid_volume = validate_ai_infra_data(
    close, volume
)

print(f'Valid symbols  : {len(valid_close.columns)}')
if skipped:
    print(f'Skipped (non-positive prices — no data fabricated): {skipped}')

# Validation summary table
val_rows = []
for sym, rep in val_report.items():
    val_rows.append({
        'symbol': sym,
        'n_bars': rep['n_bars'],
        'coverage': rep['coverage'],
        'eligible_for_selection': rep['eligible_for_selection'],
        'issues': len(rep['issues']),
        'first_issue': rep['issues'][0][:60] if rep['issues'] else '',
    })
pd.DataFrame(val_rows).set_index('symbol')

## 7. Run Alpha Search Agent Review

In [ ]:
# The pipeline uses DataEngineerAgent, OpportunityAgent, and supporting agents
# to review data quality and rank opportunities.
# This cell runs the agent layer standalone for inspection.

from alpha_search.research.ai_infra_strategy_pipeline import _run_agent_review

agent_review_text = _run_agent_review(
    valid_close, valid_volume, bench_close, val_report, skipped
)
print(agent_review_text[:2000])  # first 2000 chars

## 8. Run Strategy Families

### Pre-registered parameters (declared before any results seen):

| Parameter | Value | Rationale |
|---|---|---|
| CS momentum lookback | 252 days | Standard 12-month |
| CS momentum skip | 21 days | Avoid short-term reversal |
| MA windows (trend) | 50, 100, 200 | Classic trend filters |
| MR window | 20 days | Short mean-reversion |
| MR z-threshold | 2.0 σ | Standard 2-sigma entry |
| Breakout window | 20 days | 1-month Donchian |

In [ ]:
daily_rets = valid_close.pct_change()

# --- Cross-sectional momentum (primary — L/S dollar-neutral) ---
print('Building cross-sectional momentum signal...')
cs_weights, _ = build_cross_sectional_momentum_signal(
    valid_close, valid_volume,
    lookback=252, skip=21,
    liq_window=63, min_dollar_vol=25e6,
    quantile=1/3, freq='ME',
    long_only=LONG_ONLY,
)
print(f'  Rebalance dates: {len(cs_weights)} | avg longs: {(cs_weights > 0).sum(axis=1).replace(0, float("nan")).mean():.1f}')

# --- Trend following ---
print('Building trend-following signal...')
tf_signal = build_trend_following_signal(valid_close, ma_windows=[50, 100, 200])
tf_weights = build_monthly_rebalance_weights(tf_signal, freq='ME', long_only=True, min_eligible=4)
print(f'  Rebalance dates with positions: {(tf_weights.abs().sum(axis=1) > 0).sum()}')

# --- Mean reversion ---
print('Building mean-reversion signal...')
mr_signal = build_mean_reversion_signal(valid_close, window=20, z_threshold=2.0, allow_short=False)
mr_weights = build_monthly_rebalance_weights(mr_signal, freq='ME', long_only=True, min_eligible=4)

# --- Breakout ---
print('Building breakout signal (shift(1) prevents lookahead)...')
bo_signal = build_breakout_signal(valid_close, window=20)
bo_weights = build_monthly_rebalance_weights(bo_signal, freq='ME', long_only=True, min_eligible=4)

print('All signals built.')

## 9. Backtest with Transaction Costs and Slippage

In [ ]:
print(f'Running backtests ({TOTAL_COST_BPS:.0f} bps round-trip cost)...')

bt_cs = run_strategy_backtest(daily_rets, cs_weights, TOTAL_COST_BPS, is_dollar_neutral=not LONG_ONLY)
bt_tf = run_strategy_backtest(daily_rets, tf_weights, TOTAL_COST_BPS, is_dollar_neutral=False)
bt_mr = run_strategy_backtest(daily_rets, mr_weights, TOTAL_COST_BPS, is_dollar_neutral=False)
bt_bo = run_strategy_backtest(daily_rets, bo_weights, TOTAL_COST_BPS, is_dollar_neutral=False)

# Benchmark returns
bench_rets = {col: bench_close[col].pct_change().dropna() for col in bench_close.columns}

for name, bt in [('CS Momentum', bt_cs), ('Trend Following', bt_tf),
                  ('Mean Reversion', bt_mr), ('Breakout', bt_bo)]:
    print(f'  {name:18s} | net rebalances={bt["n_rebal"]:3d} | '
          f'turnover/rebal={bt["turnover_per_rebal"]:.3f} | cost_drag={bt["cost_drag"]:.4f}')

## 10. Compare Against QQQ, SPY, SMH, SOXX

In [ ]:
primary_bench_ret = bench_rets.get(PRIMARY_BENCH, pd.Series(dtype=float))

# Compute metrics for all strategies
METRIC_COLS = ['sharpe_ratio', 'annualized_return', 'annualized_vol', 'max_drawdown',
               'calmar_ratio', 'monthly_hit_rate', 't_stat_monthly']

strategy_metrics = {}
for name, bt in [('cs_momentum (L/S)', bt_cs), ('trend_following', bt_tf),
                  ('mean_reversion', bt_mr), ('breakout', bt_bo)]:
    m = calculate_strategy_metrics(bt['net'], rf_annual=0.0)
    m['gross_sharpe'] = calculate_strategy_metrics(bt['gross']).get('sharpe_ratio')
    ab = calculate_alpha_beta_vs_benchmark(bt['net'], primary_bench_ret)
    m.update({f'alpha_{k}': v for k, v in ab.items()})
    strategy_metrics[name] = m

for bname, bret in bench_rets.items():
    m = calculate_strategy_metrics(bret, rf_annual=0.0)
    strategy_metrics[f'bench_{bname}'] = m

metrics_df = pd.DataFrame(strategy_metrics).T
print('=== Net-of-cost performance vs benchmarks ===')
display(metrics_df[METRIC_COLS].style.format({
    'sharpe_ratio': '{:.3f}',
    'annualized_return': '{:.2%}',
    'annualized_vol': '{:.2%}',
    'max_drawdown': '{:.2%}',
    'calmar_ratio': '{:.3f}',
    'monthly_hit_rate': '{:.2%}',
    't_stat_monthly': '{:.2f}',
}).highlight_max(subset=['sharpe_ratio'], color='#c6efce'
).highlight_min(subset=['max_drawdown'], color='#ffc7ce'))
print(f'\n⚠ rf = 0.0 (no FRED). Long-only Sharpe overstated in high-rate regimes.')

## 11. Portfolio Blend

Equal-weight blend of net returns across the four strategy families.

In [ ]:
# Equal-weight blend (not re-optimized — straight average)
blend_net = pd.concat([
    bt_cs['net'], bt_tf['net'], bt_mr['net'], bt_bo['net']
], axis=1).mean(axis=1).dropna()

blend_metrics = calculate_strategy_metrics(blend_net, rf_annual=0.0)
blend_ab = calculate_alpha_beta_vs_benchmark(blend_net, primary_bench_ret)

print('=== Equal-weight strategy blend ===')
for k in ('sharpe_ratio', 'annualized_return', 'max_drawdown', 'monthly_hit_rate'):
    v = blend_metrics.get(k, float('nan'))
    print(f'  {k:25s}: {v:.4f}')
print(f'  ann_alpha_vs_{PRIMARY_BENCH}   : {blend_ab["ann_alpha"]:.4f}  (t={blend_ab["t_alpha"]:.2f})')
print(f'\nNote: blend Sharpe may look better than individual strategies.\n'
      f'This is diversification, not signal improvement. Validate out-of-sample.')

## 12. Plot Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

COLOURS = {'CS Momentum (L/S)': 'royalblue', 'Trend Following': 'seagreen',
           'Mean Reversion': 'darkorange', 'Breakout': 'purple', 'Blend': 'black'}

ax = axes[0]
for name, bt in [('CS Momentum (L/S)', bt_cs), ('Trend Following', bt_tf),
                  ('Mean Reversion', bt_mr), ('Breakout', bt_bo)]:
    bt['equity_net'].plot(ax=ax, label=name, color=COLOURS.get(name), linewidth=1.3, alpha=0.85)
(1 + blend_net).cumprod().plot(ax=ax, label='Blend (EW)', color='black', linewidth=1.5, linestyle='--')
for bname in [PRIMARY_BENCH, 'QQQ']:
    if bname in bench_rets:
        (1 + bench_rets[bname]).cumprod().plot(ax=ax, label=f'{bname} (bench)',
                                                linewidth=1, linestyle=':',  alpha=0.7)
ax.set_title('Net Equity Curves (log scale)')
ax.set_yscale('log')
ax.legend(fontsize=7)

ax = axes[1]
for name, bt in [('CS Momentum (L/S)', bt_cs), ('Trend Following', bt_tf),
                  ('Mean Reversion', bt_mr), ('Breakout', bt_bo)]:
    eq = bt['equity_net']
    dd = (eq - eq.cummax()) / eq.cummax()
    dd.plot(ax=ax, label=name, color=COLOURS.get(name), linewidth=1.1, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.7)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_title('Drawdown (negative convention)')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Rolling 63-day Sharpe
ax = axes[0]
for name, bt in [('CS Momentum (L/S)', bt_cs), ('Trend Following', bt_tf),
                  ('Mean Reversion', bt_mr), ('Breakout', bt_bo)]:
    r = bt['net']
    rs = r.rolling(63).mean() / (r.rolling(63).std() + 1e-9) * np.sqrt(252)
    rs.plot(ax=ax, label=name, color=COLOURS.get(name), linewidth=1.1, alpha=0.85)
ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax.axhline(1, color='green', linewidth=0.8, linestyle=':')
ax.set_title('Rolling 63-day Sharpe')
ax.legend(fontsize=7)

# Strategy comparison (gross vs net Sharpe)
ax = axes[1]
strats_plot = ['cs_momentum (L/S)', 'trend_following', 'mean_reversion', 'breakout']
xs = np.arange(len(strats_plot))
gross_s = [strategy_metrics[s].get('gross_sharpe', 0) for s in strats_plot]
net_s   = [strategy_metrics[s].get('sharpe_ratio', 0) for s in strats_plot]
ax.bar(xs - 0.2, gross_s, 0.35, label='Gross Sharpe', color='steelblue', alpha=0.7)
ax.bar(xs + 0.2, net_s,   0.35, label='Net Sharpe',   color='firebrick', alpha=0.7)
ax.axhline(0, color='black', linewidth=0.7)
ax.set_xticks(xs)
ax.set_xticklabels([s.replace('(', '\n(') for s in strats_plot], fontsize=7)
ax.set_title('Gross vs Net Sharpe')
ax.legend()

plt.tight_layout()
plt.show()

## 13. Export CSV Files

In [ ]:
import os
from datetime import datetime, timezone

ts = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
run_dir = os.path.join(OUTPUT_DIR, ts)
os.makedirs(run_dir, exist_ok=True)

# Strategy returns
returns_df = pd.DataFrame({
    'cs_momentum_net': bt_cs['net'],
    'trend_following_net': bt_tf['net'],
    'mean_reversion_net': bt_mr['net'],
    'breakout_net': bt_bo['net'],
    'blend_net': blend_net,
    **{f'bench_{k}': v for k, v in bench_rets.items()},
})
returns_df.to_csv(os.path.join(run_dir, 'strategy_returns.csv'))

# Metrics summary
metrics_df.to_csv(os.path.join(run_dir, 'strategy_results_summary.csv'))

# Universe
pd.DataFrame({'symbol': list(valid_close.columns)}).to_csv(
    os.path.join(run_dir, 'universe_used.csv'), index=False
)

if skipped:
    pd.DataFrame({'symbol': skipped}).to_csv(
        os.path.join(run_dir, 'skipped_symbols.csv'), index=False
    )

print(f'CSVs written to: {run_dir}')

## 14. Generate Markdown Report

In [ ]:
# Assemble minimal results dict for report generator
results_for_report = {
    'universe_used': list(valid_close.columns),
    'symbols_skipped': skipped,
    'validation_report': val_report,
    'period': PERIOD,
    'interval': INTERVAL,
    'cost_bps': TOTAL_COST_BPS,
    'long_only': LONG_ONLY,
    'rf_annual': 0.0,
    'primary_benchmark': PRIMARY_BENCH,
    'run_timestamp': datetime.now(timezone.utc).isoformat(),
    'duration_seconds': 0,
    'disclaimer': 'RESEARCH / EDUCATIONAL PURPOSES ONLY. NOT INVESTMENT ADVICE.',
    'agent_review': agent_review_text,
    'strategies': {
        'cross_sectional_momentum': {
            'verdict': 'see metrics below',
            'metrics_net': strategy_metrics.get('cs_momentum (L/S)', {}),
            'metrics_gross': {'sharpe_ratio': strategy_metrics.get('cs_momentum (L/S)', {}).get('gross_sharpe')},
            'alpha_beta': {k.replace('alpha_', ''): v for k, v in strategy_metrics.get('cs_momentum (L/S)', {}).items() if k.startswith('alpha_')},
            'is_metrics': {}, 'oos_metrics': {},
            'hypothesis': 'Cross-sectional 12-1 momentum, monthly rebalance, L/S dollar-neutral.',
            'is_primary': True,
            'backtest': {'net': bt_cs['net'], 'gross': bt_cs['gross'], 'equity_net': bt_cs['equity_net'],
                         'equity_gross': bt_cs['equity_gross'], 'turnover_per_rebal': bt_cs['turnover_per_rebal'],
                         'cost_drag': bt_cs['cost_drag'], 'n_rebal': bt_cs['n_rebal']},
        },
    },
    'bench_summary': {bname: calculate_strategy_metrics(bret) for bname, bret in bench_rets.items()},
}

report_path = os.path.join(run_dir, 'report.md')
generate_ai_infra_report(results_for_report, report_path)
print(f'Report written to: {report_path}')

with open(report_path) as f:
    print(f.read()[:3000])

## 15. Honest Conclusion

In [ ]:
print('=' * 65)
print('  HONEST RESEARCH CONCLUSION')
print('=' * 65)
print(f'  Universe: {len(valid_close.columns)} symbols | Period: {PERIOD}')
if skipped:
    print(f'  Skipped:  {skipped} (non-positive prices)')
print()

_VERDICT_MAP = {
    'promising':        'Sharpe > 1.0 and alpha t-stat > 2 — worth further stress-testing',
    'marginal_positive':'0.5 < Sharpe ≤ 1.0 — positive edge, cost-sensitive',
    'marginal':         '0 < Sharpe ≤ 0.5 — weak positive edge',
    'unprofitable':     'Sharpe ≤ 0 — strategy lost money after costs',
    'no_results':       'No trades generated',
}

for name, bt in [('CS Momentum (L/S)', bt_cs), ('Trend Following', bt_tf),
                  ('Mean Reversion', bt_mr), ('Breakout', bt_bo)]:
    m = calculate_strategy_metrics(bt['net'])
    sr = m.get('sharpe_ratio', float('nan'))
    ab = calculate_alpha_beta_vs_benchmark(bt['net'], primary_bench_ret)
    ta = ab.get('t_alpha', float('nan'))
    if np.isnan(sr):           verdict = 'no_results'
    elif sr > 1.0 and abs(ta) >= 2.0: verdict = 'promising'
    elif sr > 0.5:             verdict = 'marginal_positive'
    elif sr > 0.0:             verdict = 'marginal'
    else:                      verdict = 'unprofitable'
    desc = _VERDICT_MAP[verdict]
    print(f'  {name:22s}  Sharpe={sr:6.3f}  t(α)={ta:5.2f}  → {verdict}')
    print(f'    {desc}')
    print()

print('  Suggested next steps:')
print('  1. Walk-forward validation (rolling IS/OOS windows).')
print('  2. Regime filter (VIX > 30 = reduce exposure).')
print('  3. Parameter sensitivity: vary lookback ±50 days.')
print('  4. Survivorship-bias correction (add delisted names).')
print('  5. Liquidity-weighted position sizing.')
print()
print('  ⚠  REMINDER: Research only. NOT investment advice.')
print('  ⚠  rf = 0.0 (no FRED) — long-only Sharpe overstated in high-rate regimes.')
print('=' * 65)

---

## Bonus: One-Line Full Pipeline

Equivalent to everything above, including agent review, all four strategies, export, and report.

In [ ]:
# Uncomment to run:

# results = run_ai_infra_research(
#     period='5y',
#     interval='1d',
#     cost_bps=10.0,
#     slippage_bps=10.0,
#     long_only=False,
#     output_dir='outputs/research_runs/ai_infrastructure',
# )
#
# print('CS momentum verdict :', results['strategies']['cross_sectional_momentum']['verdict'])
# print('Output dir          :', results['output_dir'])

# CLI equivalent (run in terminal or Colab shell):
# !python scripts/run_ai_infra_research.py --period 5y --interval 1d --cost-bps 10 --slippage-bps 10

print('One-liner commented out to avoid duplicate yfinance downloads.')
print('Uncomment the block above to run the full automated pipeline.')